-*- coding: utf-8 -*-


"""
=====================================================================
[공동작업] ESG 보고서 인덱스 구축 - 기업당 딱 1번만 실행
- PDF Parsing → Chunking(800 tokens) → BM25(Kiwi) → OpenAI 임베딩 → FAISS 인덱스 저장
- 팀 확정 역할 분리: OpenAI = 임베딩 전용, Gemini = 채점/챗봇 엔진(stage4에서 사용),
  Claude = 챗봇 인용 전용. 이 스크립트는 임베딩 단계라 OpenAI만 사용합니다.
- 벡터DB는 FAISS를 사용합니다 (Chroma의 HNSW 인덱스가 Google Drive/FUSE 환경에서
  반복적으로 손상되는 문제가 있어 교체). FAISS 인덱스는 파일 하나로 저장/로드되어
  Drive(FUSE)에서도 안전합니다.
- 노션 9/1 회의록: "인덱스를 팀원별로 따로 만들면 재작업 발생, 반드시 공동으로
  먼저 완성한 뒤 개인 파트(E/S/G 채점)를 시작해야 함"
- 이 스크립트가 끝나면, E/S/G 담당자는 각자 stage4_E.py / stage4_S.py / stage4_G.py를
  독립적으로 실행하면 됩니다 (이 스크립트를 다시 돌릴 필요 없음).

⚠️ 주의: stage4에서 채점용으로 Gemini를 사용하므로, 그쪽은 여전히 무료 티어
   일일 1,500건 한도(노션 9/1 회의록)를 확인하세요. (이 스크립트 자체는 OpenAI만 사용)

코랩 노트북에 셀 단위(# ===== N. ... =====)로 나눠서 붙여넣으시면 됩니다.


In [1]:
# ============================================================
# ===== 0. 환경설정 (모든 설치 + 마운트 + 경로 + 클라이언트) =====
# ============================================================
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pdfplumber", "tiktoken", "kiwipiepy", "rank_bm25",
    "openai", "openpyxl", "pandas", "numpy", "faiss-cpu",
])

import os
import re
import json
import hashlib
import pickle
from getpass import getpass
from datetime import datetime, timezone

import numpy as np
import faiss
import pdfplumber
import pandas as pd
import tiktoken
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from openai import OpenAI
import openpyxl

from google.colab import drive
drive.mount('/content/drive')

# ---- 대상 기업 설정 ----
COMPANY_CODE = "005930"
COMPANY_NAME = "삼성전자"
YEAR = 2025
IS_C261 = True  # 반도체(KSIC 261) 여부 - 반도체 특화 6개 항목 적용 판단용

# ---- 경로 설정 (팀 확정 경로) ----
REPORT_ROOT = "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종"
RUBRIC_ROOT = "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/채점 시트"

folder_path = f'{REPORT_ROOT}/{COMPANY_NAME}({COMPANY_CODE})'  # 보고서 3종 + 산출물 저장 위치

# ---- FAISS 인덱스 저장 경로 ----
# 인덱스 전체가 파일 하나(.index)로 저장되어, zip처럼 "한 번에 쭉 쓰는" 방식이라
# Google Drive(FUSE)에서도 안전합니다. (Chroma처럼 여러 파일에 걸친 랜덤 액세스가 없음)
FAISS_INDEX_PATH = f'{folder_path}/{COMPANY_CODE}_{YEAR}_faiss.index'

# ---- 임베딩 모델 (팀 확정: OpenAI 전용) ----
OPENAI_API_KEY = getpass("OpenAI API Key를 입력하세요: ")
client = OpenAI(api_key=OPENAI_API_KEY)

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536  # text-embedding-3-small 기본 차원

TOKENIZER = tiktoken.get_encoding("cl100k_base")  # 비용 견적용 근사 토크나이저 (OpenAI 계열이라 이번엔 실제 모델과 정확히 일치)
kiwi = Kiwi()

print("환경설정 완료.")
print(f"  - 보고서 경로: {folder_path}")

Mounted at /content/drive
OpenAI API Key를 입력하세요: ··········
환경설정 완료.
  - 보고서 경로: /content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종/삼성전자(005930)


In [2]:
# ============================================================
# ===== 1. PDF Parsing + Chunking (Text/Table, 800 tokens) =====
# ============================================================
pattern_keywords = {
    "governance": ["지배구조", "governance"],
    "sustainability": ["지속가능", "sustainability"],
    "business": ["사업", "business"],
}

files_2025 = []
unmatched_files = []

if os.path.exists(folder_path):
    for f in sorted(os.listdir(folder_path)):
        if not (f.lower().endswith('.pdf') and str(YEAR) in f):
            continue
        doc_type = None
        for dtype, keywords in pattern_keywords.items():
            if any(k in f or k in f.lower() for k in keywords):
                doc_type = dtype
                break
        if doc_type:
            files_2025.append({"filename": f, "doc_type": doc_type})
        else:
            unmatched_files.append(f)

    print(f"선택된 {YEAR}년 보고서 목록:")
    for item in files_2025:
        print(f"  - {item['doc_type']}: {item['filename']}")
    if unmatched_files:
        print(f"\n⚠️ doc_type 키워드를 못 찾아 제외된 파일: {unmatched_files}")
    if len(files_2025) != 3:
        missing = {"business", "governance", "sustainability"} - {i["doc_type"] for i in files_2025}
        print(f"\n⚠️ 경고: 3종 중 {len(files_2025)}개만 발견됨. 누락: {missing}")
else:
    print(f"❌ 경로를 찾을 수 없습니다: {folder_path}")


CHUNK_SIZE_TOKENS = 800
CHUNK_OVERLAP_TOKENS = 100


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE_TOKENS, overlap: int = CHUNK_OVERLAP_TOKENS) -> list[str]:
    tokens = TOKENIZER.encode(text)
    chunks, start, n = [], 0, len(tokens)
    while start < n:
        chunks.append(TOKENIZER.decode(tokens[start:start + chunk_size]))
        start += chunk_size - overlap
    return chunks


def table_to_markdown(table: list) -> str:
    if not table or not table[0]:
        return ""
    rows = [[(cell if cell is not None else "").strip() for cell in row] for row in table]
    header = rows[0]
    md = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    for row in rows[1:]:
        md.append("| " + " | ".join(row) + " |")
    return "\n".join(md)


def file_hash(path: str) -> str:
    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest()[:16]


records = []
for item in files_2025:
    doc_type = item["doc_type"]
    filepath = os.path.join(folder_path, item["filename"])
    doc_id = f"{COMPANY_CODE}_{YEAR}_{doc_type}"
    src_hash = file_hash(filepath)
    chunk_seq = 0

    print(f"\n처리 중: {doc_type} ({item['filename']})")
    with pdfplumber.open(filepath) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                md_table = table_to_markdown(table)
                if not md_table.strip():
                    continue
                chunk_seq += 1
                records.append({
                    "chunk_id": f"{doc_id}_{chunk_seq:04d}", "doc_id": doc_id,
                    "company_code": COMPANY_CODE, "company_name": COMPANY_NAME, "year": YEAR,
                    "doc_type": doc_type, "page": page_num,
                    "section_title": None,              # 1차 인덱싱 스킵 (노션 확정)
                    "content_type": "table",
                    "text_markdown": md_table,
                    "table_data": json.dumps(table, ensure_ascii=False),
                    "esg_area": None,                    # 1차 인덱싱 스킵 (노션 확정)
                    "applicable_item_codes": "[]",        # 1차 인덱싱 스킵 (노션 확정)
                    "chunk_char_count": len(md_table),
                    "chunk_token_count": len(TOKENIZER.encode(md_table)),
                    "embedding_model": None,             # Stage 3 임베딩 완료 후 채워짐
                    "indexed_at": None,                  # Stage 3 적재 시각 기록됨
                    "source_file_hash": src_hash,
                })

            text = page.extract_text() or ""
            if text.strip():
                for chunk in chunk_text(text):
                    if not chunk.strip():
                        continue
                    chunk_seq += 1
                    records.append({
                        "chunk_id": f"{doc_id}_{chunk_seq:04d}", "doc_id": doc_id,
                        "company_code": COMPANY_CODE, "company_name": COMPANY_NAME, "year": YEAR,
                        "doc_type": doc_type, "page": page_num,
                        "section_title": None,
                        "content_type": "text",
                        "text_markdown": chunk, "table_data": None,
                        "esg_area": None,
                        "applicable_item_codes": "[]",
                        "chunk_char_count": len(chunk),
                        "chunk_token_count": len(TOKENIZER.encode(chunk)),
                        "embedding_model": None,
                        "indexed_at": None,
                        "source_file_hash": src_hash,
                    })
    print(f"  → {chunk_seq}개 청크 생성")

chunks_df = pd.DataFrame(records)
csv_path = f'{folder_path}/{COMPANY_CODE}_{YEAR}_chunks.csv'
chunks_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"\n총 {len(chunks_df)}개 청크 생성 → {csv_path} 저장 완료")

선택된 2025년 보고서 목록:
  - business: 005930_2025_business.pdf
  - governance: 005930_2025_governance.pdf
  - sustainability: 005930_2025_sustainability.pdf

처리 중: business (005930_2025_business.pdf)
  → 1886개 청크 생성

처리 중: governance (005930_2025_governance.pdf)


  → 495개 청크 생성

처리 중: sustainability (005930_2025_sustainability.pdf)
  → 384개 청크 생성

총 2765개 청크 생성 → /content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종/삼성전자(005930)/005930_2025_chunks.csv 저장 완료


In [3]:
# ============================================================
# ===== 2. BM25 인덱스 구축 (Kiwi, 비용 0원) =====
# ============================================================
def tokenize_korean(text: str) -> list[str]:
    keep_tags = {"NNG", "NNP", "NNB", "VV", "VA", "SL", "SN", "SH"}
    return [t.form for t in kiwi.tokenize(text) if t.tag in keep_tags]


print("\nKiwi 형태소분석 진행 중...")
chunks_df["tokens"] = chunks_df["text_markdown"].fillna("").apply(tokenize_korean)

corpus_tokens = chunks_df["tokens"].tolist()
bm25_chunk_ids = chunks_df["chunk_id"].tolist()
bm25 = BM25Okapi(corpus_tokens)

bm25_index_path = f'{folder_path}/{COMPANY_CODE}_{YEAR}_bm25_index.pkl'
with open(bm25_index_path, "wb") as f:
    pickle.dump({"bm25": bm25, "chunk_ids": bm25_chunk_ids}, f)
print(f"BM25 인덱스 구축 완료 → {bm25_index_path}")


Kiwi 형태소분석 진행 중...
BM25 인덱스 구축 완료 → /content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종/삼성전자(005930)/005930_2025_bm25_index.pkl


In [4]:
# ============================================================
# ===== 3. OpenAI 임베딩 + FAISS 적재 (비용 발생 구간) =====
# ============================================================
def openai_embed(text_list: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text_list)
    return [item.embedding for item in response.data]


texts = chunks_df["text_markdown"].fillna("").tolist()
total_tokens = sum(len(TOKENIZER.encode(t)) for t in texts)
EMBED_PRICE_PER_1M = 0.02  # text-embedding-3-small, 2026.08 기준
est_cost = total_tokens / 1_000_000 * EMBED_PRICE_PER_1M

print(f"\n[임베딩 예상 비용] 총 {total_tokens:,} tokens → 약 ${est_cost:.4f} (약 {est_cost*1400:.0f}원)")
print("  ※ 실제 청구 금액은 OpenAI 대시보드에서 재확인하세요 (가격은 변동될 수 있음)")
if input("진행하시겠습니까? (y/n): ").lower() != "y":
    raise SystemExit("사용자가 중단했습니다.")

BATCH_SIZE = 100  # OpenAI 임베딩은 요청당 다건 처리에 강함, 넉넉히 100개씩
all_embeddings = []
print(f"\n임베딩 실행 중... (총 {len(texts)}개)")
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    all_embeddings.extend(openai_embed(batch))
    print(f"  {min(i + BATCH_SIZE, len(texts))}/{len(texts)} 완료")

indexed_at = datetime.now(timezone.utc).isoformat()  # 스테일 청크 판별용 (노션 확정 필드)
chunks_df["embedding_model"] = EMBEDDING_MODEL
chunks_df["indexed_at"] = indexed_at

# ---- FAISS 인덱스 생성 및 저장 ----
# chunks_df의 행 순서 == texts 순서 == all_embeddings 순서 == FAISS 내부 위치(0,1,2...)
# 이 순서만 유지되면, 검색 결과로 나온 FAISS 위치(정수)를 CSV의 같은 행번호로 그대로 매칭할 수 있습니다.
embeddings_array = np.array(all_embeddings, dtype=np.float32)

# 코사인 유사도로 검색하기 위해 벡터를 정규화한 뒤, 내적(Inner Product) 인덱스를 사용합니다.
# (정규화된 벡터끼리의 내적 = 코사인 유사도와 동일)
faiss.normalize_L2(embeddings_array)
faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
faiss_index.add(embeddings_array)

faiss.write_index(faiss_index, FAISS_INDEX_PATH)

print(f"\nFAISS 인덱스 저장 완료: {faiss_index.ntotal}개 벡터 → {FAISS_INDEX_PATH}")
if faiss_index.ntotal != len(chunks_df):
    print(f"⚠️ 경고: 인덱스 벡터 개수({faiss_index.ntotal})와 청크 개수({len(chunks_df)})가 다릅니다. "
          f"임베딩 루프가 중간에 끊겼을 수 있으니 이 셀을 다시 실행하세요.")

# 임베딩 완료 메타데이터를 CSV에도 반영해서 재저장 (추적용, indexed_at/embedding_model 포함)
# ※ 이 CSV가 곧 "메타데이터 저장소" 역할을 합니다 (행 순서로 FAISS 인덱스와 1:1 대응)
chunks_df.drop(columns=["tokens"], errors="ignore").to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"임베딩 완료 메타데이터 CSV 갱신: {csv_path}")


[임베딩 예상 비용] 총 1,634,771 tokens → 약 $0.0327 (약 46원)
  ※ 실제 청구 금액은 OpenAI 대시보드에서 재확인하세요 (가격은 변동될 수 있음)
진행하시겠습니까? (y/n): y

임베딩 실행 중... (총 2765개)
  100/2765 완료
  200/2765 완료
  300/2765 완료
  400/2765 완료
  500/2765 완료
  600/2765 완료
  700/2765 완료
  800/2765 완료
  900/2765 완료
  1000/2765 완료
  1100/2765 완료
  1200/2765 완료
  1300/2765 완료
  1400/2765 완료
  1500/2765 완료
  1600/2765 완료
  1700/2765 완료
  1800/2765 완료
  1900/2765 완료
  2000/2765 완료
  2100/2765 완료
  2200/2765 완료
  2300/2765 완료
  2400/2765 완료
  2500/2765 완료
  2600/2765 완료
  2700/2765 완료
  2765/2765 완료

FAISS 인덱스 저장 완료: 2765개 벡터 → /content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종/삼성전자(005930)/005930_2025_faiss.index
임베딩 완료 메타데이터 CSV 갱신: /content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/보고서 3종/삼성전자(005930)/005930_2025_chunks.csv
